In [10]:
import os
import pickle
import pandas as pd
from pathlib import Path
from tqdm import tqdm

In [ ]:
# ================= 配置区 =================
ROOT_DIRS = ["../result/archive-103"] 
OUTPUT_FILE_LOC = "./result_101/experiment_results_PowerBI.csv"
TARGET_BUDGETS = {6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0}

# 允许被解析的 EfficientBFS 策略类型 (自由调整区)
ALLOWED_STRATEGIES = {'traditional', 'lazy_binary', 'binary_collapse'} 
# ==========================================

def process_experiment_data(root_paths):
    data_records = []
    pckl_files = []
    
    for r in root_paths:
        path_obj = Path(r)
        if path_obj.exists():
            pckl_files.extend(list(path_obj.rglob("*.pckl")))
        else:
            print(f"⚠️ 警告: 找不到目录 {r}")
            
    print(f"🔍 总共寻找到 {len(pckl_files)} 个 pickle 文件，开始解析...")
    
    for pckl_file in tqdm(pckl_files, desc="Parsing Files"):
        try:
            seed_str = pckl_file.parent.name
            num_str = pckl_file.parent.parent.name
            task_name = pckl_file.parent.parent.parent.name
            
            if task_name == 'caltech' and num_str == '767':
                continue
            
            filename = pckl_file.stem
            parts = filename.split('-')
            
            record = {
                'Task': task_name,
                'Ground_Size': int(num_str),
                'Seed': int(seed_str),
                'Algorithm': None,
                'Strategy': None,
                'UB': None,
                'D': None,
                'Budget': None,
                'Alpha': None,
                'Model': None,
            }

            try:
                budget = float(parts[-3])
                alpha = float(parts[-2])
                model = parts[-1]
                
                if not any(abs(budget - t) < 1e-4 for t in TARGET_BUDGETS):
                    continue
                
                record['Budget'] = budget
                record['Alpha'] = alpha
                record['Model'] = model

                if parts[0] == 'EfficientBFS':
                    strategy = parts[1]
                    
                    # 策略过滤
                    if strategy not in ALLOWED_STRATEGIES:
                        continue 
                    
                    # ========= 算法命名映射 =========
                    if strategy == 'traditional' and parts[2] == 'ub0':
                        record['Algorithm'] = 'simple'
                    elif strategy == 'lazy_binary':
                        record['Algorithm'] = 'lazy_binary'
                    elif strategy == 'binary_collapse':
                        record['Algorithm'] = 'binary_collapse'
                    else:
                        record['Algorithm'] = 'EfficientBFS'
                    # ================================
                    
                    record['Strategy'] = strategy
                    record['UB'] = parts[2]
                    record['D'] = parts[3]
                    
                elif parts[0] in ['BFSTC', 'Efficient', 'AdaptiveEfficientBFS']:
                    record['Algorithm'] = parts[0]
                    record['UB'] = parts[1]
                    record['D'] = parts[2]
                else:
                    continue
                    
            except (ValueError, IndexError):
                continue

            with open(pckl_file, 'rb') as f:
                res = pickle.load(f)
                
            raw_time = res.get('time', None)
            raw_tle = res.get('TLE', False)
            
            if raw_time is not None and raw_time >= 5000:
                raw_time = 5000
                raw_tle = True
                
            record.update({
                'Objective_f(S)': res.get('f(S)', None),
                'Cost_c(S)': res.get('c(S)', None),
                'Time_s': raw_time,
                'Node_Count': res.get('node_count', None),
                'Open_List_Count': res.get('open_list_count', None),
                'TLE': raw_tle,
                'Solution_Set_Size': len(res.get('S', [])) if 'S' in res else 0 
            })
            
            data_records.append(record)
            
        except Exception as e:
            print(f"❌ 解析出错 {pckl_file.name}: {e}")

    df = pd.DataFrame(data_records)
    if not df.empty:
        df.sort_values(by=['Task', 'Algorithm', 'UB', 'Budget', 'Seed'], inplace=True)
        Path(OUTPUT_FILE_LOC).parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(OUTPUT_FILE_LOC, index=False, encoding='utf-8-sig')
        print(f"\n✅ 处理完毕！提取 {len(df)} 条记录。")
        print(f"💾 保存至: {OUTPUT_FILE_LOC}")
    else:
        print("\n⚠️ 未找到匹配数据，请检查路径及 TARGET_BUDGETS。")
        
    return df

if __name__ == "__main__":
    df_results = process_experiment_data(ROOT_DIRS)
    if not df_results.empty:
        print(df_results.head())

🔍 总共寻找到 221 个 pickle 文件，开始解析...


Parsing Files: 100%|██████████| 221/221 [00:00<00:00, 313.42it/s]



✅ 处理完毕！提取 211 条记录。
💾 保存至: ./result_101/experiment_results_PowerBI.csv
    Task  Ground_Size  Seed Algorithm Strategy   UB  D  Budget  Alpha  \
6  adult          111     0     BFSTC     None  ub0  d     6.0   0.95   
7  adult          111     0     BFSTC     None  ub0  d     7.0   0.95   
8  adult          111     0     BFSTC     None  ub0  d     8.0   0.95   
9  adult          111     0     BFSTC     None  ub0  d     9.0   0.95   
0  adult          111     0     BFSTC     None  ub0  d    10.0   0.95   

                         Model  Objective_f(S)  Cost_c(S)  Time_s  Node_Count  \
6  AdultIncomeFeatureSelection        7.206555   5.836918  5000.0        1568   
7  AdultIncomeFeatureSelection        7.206555   6.999571  5000.0         100   
8  AdultIncomeFeatureSelection        8.120308   7.606156  5000.0          79   
9  AdultIncomeFeatureSelection        8.270787   8.931538  5000.0          72   
0  AdultIncomeFeatureSelection        8.655823   9.305434  5000.0          71   

   

In [15]:
import os
from pathlib import Path

# ================= 配置区 =================
# 根据你的截图，设置根目录路径。注意使用原始字符串 r"..." 防止反斜杠转义问题
ROOT_DIR = r"D:\archive-81\archive-81" 

OLD_PREFIX = "EfficientBFS"
NEW_PREFIX = "AdaptiveEfficientBFS"
# ==========================================

def rename_files(root_path, dry_run=True):
    target_dir = Path(root_path)

    if not target_dir.exists():
        print(f"❌ 找不到目录: {root_path}")
        return

    # rglob("*.pckl") 会递归寻找所有子文件夹下的 .pckl 文件
    pckl_files = list(target_dir.rglob("*.pckl"))
    print(f"🔍 总共找到 {len(pckl_files)} 个 .pckl 文件，开始检查...")

    rename_count = 0
    for pckl_file in pckl_files:
        old_name = pckl_file.name

        # 确保只修改前缀匹配的文件，避免误伤其他包含该字符串的文件
        if old_name.startswith(OLD_PREFIX + "-"):
            # 替换旧的前缀为新的前缀，并限制只替换1次
            new_name = old_name.replace(OLD_PREFIX, NEW_PREFIX, 1)
            new_file_path = pckl_file.parent / new_name

            if dry_run:
                print(f"[预览] 准备重命名: {old_name}  ->  {new_name}")
            else:
                try:
                    pckl_file.rename(new_file_path)
                    print(f"✅ 成功重命名: {old_name}  ->  {new_name}")
                except Exception as e:
                    print(f"❌ 重命名失败 {old_name}: {e}")

            rename_count += 1

    print("-" * 50)
    if dry_run:
        print(f"💡 [预览模式结束] 共有 {rename_count} 个文件符合修改条件。")
        print("👉 如果上方打印出的修改方案没有问题，请将代码末尾的 `dry_run=True` 改为 `dry_run=False` 并重新运行。")
    else:
        print(f"🎉 [重命名完成] 成功修改了 {rename_count} 个文件名！")

if __name__ == "__main__":
    # 第一步：先保持 dry_run=True 运行一次，确认打印出的替换路径是对的。
    # 第二步：确认无误后，将这里的 True 改成 False，再运行一次即可。
    rename_files(ROOT_DIR, dry_run=False)

🔍 总共找到 40 个 .pckl 文件，开始检查...
✅ 成功重命名: EfficientBFS-traditional-ub2-d-10.0-0.95-AdultIncomeFeatureSelection.pckl  ->  AdaptiveEfficientBFS-traditional-ub2-d-10.0-0.95-AdultIncomeFeatureSelection.pckl
✅ 成功重命名: EfficientBFS-traditional-ub2-d-11.0-0.95-AdultIncomeFeatureSelection.pckl  ->  AdaptiveEfficientBFS-traditional-ub2-d-11.0-0.95-AdultIncomeFeatureSelection.pckl
✅ 成功重命名: EfficientBFS-traditional-ub2-d-12.0-0.95-AdultIncomeFeatureSelection.pckl  ->  AdaptiveEfficientBFS-traditional-ub2-d-12.0-0.95-AdultIncomeFeatureSelection.pckl
✅ 成功重命名: EfficientBFS-traditional-ub2-d-13.0-0.95-AdultIncomeFeatureSelection.pckl  ->  AdaptiveEfficientBFS-traditional-ub2-d-13.0-0.95-AdultIncomeFeatureSelection.pckl
✅ 成功重命名: EfficientBFS-traditional-ub2-d-14.0-0.95-AdultIncomeFeatureSelection.pckl  ->  AdaptiveEfficientBFS-traditional-ub2-d-14.0-0.95-AdultIncomeFeatureSelection.pckl
✅ 成功重命名: EfficientBFS-traditional-ub2-d-15.0-0.95-AdultIncomeFeatureSelection.pckl  ->  AdaptiveEfficientBFS-traditional-